# Descriptive Statistics and Collinearity Assessment

**Author:** Ali Farzaneh

This notebook generates descriptive characteristics of the analytical study population and assesses multicollinearity among covariates used in the statistical analyses.

## Load required packages

In [1]:
libs <- c("dplyr", "purrr", "survival", "car")

suppressPackageStartupMessages(
   invisible( lapply(libs, library, character.only = TRUE))
)

Warning message:
"package 'car' was built under R version 4.6.1"
Warning message:
"package 'carData' was built under R version 4.6.1"


## Study Population
Baseline corresponds to the visit at which circulating miRNAs were measured (RS-I-4 or RS-II-2). Participants with prevalent HF at this baseline
or unknown baseline HF status are excluded.

In [2]:
output_dir = 'V:/Uitwissel/Ali/Lotte/microRNA/Results_v6/'
analysis_data <- read.csv('Z:/051379(Ali_Farzaneh)/microRNA/out_dat.csv') %>% 
    filter( prevalent_hf == 0)

# data_path <- "data/analysis_data.csv"
# output_dir <- "results/"

# out_dat <- read.csv(data_path) %>%
#     filter( prevalant_hf == 0)
dim(analysis_data)

[1] 1897   54

## Functions

In [3]:
# Summary of categorical variables
summary_categorical <- function(x) {
    n <- sum(!is.na(x))
    
    n_1 <- sum(x == 1, na.rm = TRUE)
    pct_1 <- n_1 / n * 100
    
    n_2 <- sum(x == 2, na.rm = TRUE)
    pct_2 <- n_2 / n * 100
    
    if (n_2 == 0) {
        tibble::tibble(
            Number_Percentage = paste0(
                n_1, " (", round(pct_1), "%)"
            )
        )
    } else {
        tibble::tibble(
            Number_Percentage = paste0(
                "1: ", n_1, " (", round(pct_1), "%); ",
                "2: ", n_2, " (", round(pct_2), "%)"
            )
        )
    }
}

# Summary of continuous variables
summary_continuous <- function(x) {
    n <- sum(!is.na(x))
    mean_value <- round(mean(x, na.rm = TRUE), 2)
    sd_value <- round(sd(x, na.rm = TRUE), 2)
    median_value <- round(median(x, na.rm = TRUE), 2)
    q1 <- round(quantile(x, 0.25, na.rm = TRUE), 2)
    q3 <- round(quantile(x, 0.75, na.rm = TRUE), 2)
    min_value <- round(min(x, na.rm = TRUE), 2)
    max_value <- round(max(x, na.rm = TRUE), 2)
    
    tibble::tibble(
        Number = n,
        Mean_SD = paste0(mean_value, " (", sd_value, ")"),
        Median = median_value,
        Quartile_1 = q1,
        Quartile_3 = q3,
        Min = min_value,
        Max = max_value,
        Median_Q1_Q3 = paste0(
            median_value, " (", q1, " - ", q3, ")"
        )
    )
}

### Descriptive analysis

In [4]:
# Covariates included in the adjusted models
# WBC and RBC are included to account for immune/inflammatory
# and hematologic variation, respectively.

covariates <- c(
    "sex", "age", "bmi", "wbc_c", "rbc_c", "eGFR",
    "prevalent_dm", "HT", "tc", "hdl", "smoking",
    "lipid_reducing"
)

categorical_covariates <- c(
    "sex", "prevalent_dm", "HT", "smoking", "lipid_reducing"
)

continuous_covariates <- setdiff(
    covariates,
    categorical_covariates
)


In [5]:
# Descriptive statistics for categorical variables
# Smoking coding: 0 = never, 1 = current, 2 = former

desc_cat <- analysis_data %>%
    select(all_of(categorical_covariates)) %>%
    lapply(summary_categorical) %>%
    map_dfr( ~.x,.id = "Variable")

desc_cat

write.csv(
    desc_cat,
    file.path(output_dir, "description_categorical.csv"),
    row.names = FALSE
)

Variable,Number_Percentage
<chr>,<chr>
sex,1090 (57%)
prevalent_dm,256 (13%)
HT,1458 (77%)
smoking,1: 257 (14%); 2: 1018 (54%)
lipid_reducing,404 (21%)


In [6]:
# Descriptive statistics for continuous variables

desc_con <- analysis_data %>%
    select(all_of(continuous_covariates)) %>%
    lapply(summary_continuous) %>%
    map_dfr( ~.x, .id = "Variable")

desc_con

write.csv(
    desc_con,
    file.path(output_dir, "description_continuous.csv"),
    row.names = FALSE
)

Variable,Number,Mean_SD,Median,Quartile_1,Quartile_3,Min,Max,Median_Q1_Q3
<chr>,<int>,<chr>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<chr>
age,1897,71.38 (7.43),70.19,65.45,76.58,58.21,97.97,70.19 (65.45 - 76.58)
bmi,1860,27.62 (4.1),27.21,24.84,29.97,15.28,50.33,27.21 (24.84 - 29.97)
wbc_c,1894,6.77 (3.17),6.40,5.40,7.50,1.70,90.80,6.4 (5.4 - 7.5)
rbc_c,1895,4.73 (0.44),4.72,4.46,5.00,2.41,9.30,4.72 (4.46 - 5)
eGFR,1890,75.81 (14.34),76.33,66.34,86.68,15.49,106.82,76.33 (66.34 - 86.68)
tc,1897,5.67 (0.98),5.65,5.03,6.28,2.60,9.32,5.65 (5.03 - 6.28)
hdl,1897,1.45 (0.4),1.40,1.16,1.69,0.26,3.59,1.4 (1.16 - 1.69)


## Incident Heart Failure and Follow-up

This section summarizes the number of incident heart failure events, total follow-up time, incidence rate with 95% confidence interval, cumulative incidence, and median follow-up time in the analytical study population.

In [7]:
# Number of incident HF cases
num_incident_cases <- sum(
    analysis_data$incident_hf == 1,
    na.rm = TRUE
)
sprintf ("incident HF cases : %s",num_incident_cases)
# Total follow-up time in person-years
total_person_years <- sum(
    analysis_data$obstime_hf,
    na.rm = TRUE
)
sprintf ("total person years : %s", round(total_person_years, 2))
# Incidence rate per 1,000 person-years
incidence_rate <- (
    num_incident_cases / total_person_years
) * 1000

# 95% Poisson confidence interval for the incidence rate
lower_ci <- (
    qchisq(0.025, 2 * num_incident_cases) /
    (2 * total_person_years)
) * 1000

upper_ci <- (
    qchisq(0.975, 2 * (num_incident_cases + 1)) /
    (2 * total_person_years)
) * 1000
sprintf ("incidence rate per 1000_PY : %s (lower 95 CI : %s, upper 95 CI : %s )", 
         round(incidence_rate, 2),round(lower_ci, 2),round(upper_ci, 2))

# Cumulative incidence (%)
cumulative_incidence <- ( num_incident_cases / nrow(analysis_data)) * 100

sprintf ("cumulative incidence percent :  %s", round(cumulative_incidence, 2))

# Median follow-up using the reverse Kaplan-Meier method
# HF events are censored; non-HF censoring times define the follow-up distribution.

reverse_km <- survfit(
    Surv(obstime_hf, 1 - incident_hf) ~ 1,
    data = analysis_data
)

median_followup <- summary(reverse_km)$table["median"]
sprintf ("median followup years by reverse Kaplan-Meier : %s", round(median_followup, 2))

# Simple median of observed follow-up
median_followup_simple <- median(analysis_data$obstime_hf, na.rm = TRUE)
sprintf ("median followup years : %s", round(median_followup_simple, 2))

[1] "incident HF cases : 316"

[1] "total person years : 22245.81"

[1] "incidence rate per 1000_PY : 14.2 (lower 95 CI : 12.68, upper 95 CI : 15.86 )"

[1] "cumulative incidence percent :  16.66"

[1] "median followup years by reverse Kaplan-Meier : 14.75"

[1] "median followup years : 14.24"